# 06 — Run the Trained Model on the ARC-AGI Benchmark

This is the "moment of truth" notebook: take whatever adapter you produced from notebook `02_train_h100` or `05_train_eval_interp`, point it at the **actual ARC-AGI public eval set**, and get a real, comparable score.

We'll go step by step:

1. Pick where the adapter lives (local disk / HuggingFace Hub / a zipped run bundle).
2. Pull the official ARC-AGI eval data.
3. Load base model + adapter on whatever GPU you have.
4. Run the benchmark **3 ways**, fastest → strongest:
   - **Greedy** (one shot, ~1 min/task on T4)
   - **D4 self-consistency vote** (the standard +5–10 pt trick used by every public top fine-tune)
   - **Test-time training (TTT)** (the +15–25 pt trick — slowest, optional)
5. Generate the official `submission.json`.
6. Inspect what worked and what didn't (interactive plotly).
7. (Optional) Push the adapter to HuggingFace Hub so you can host it anywhere.
8. (Optional) Serve it locally with vLLM.
9. Submit to the real ARC Prize leaderboard via Kaggle.

Designed to run on Colab (T4 free / L4 Pro / H100 Pro+) or a local GPU box.


## What is the ARC-AGI benchmark?

**ARC-AGI** = *Abstraction and Reasoning Corpus*, by François Chollet (2019). Each task is a tiny visual reasoning puzzle: 2–4 input/output grid examples that demonstrate a transformation rule, plus a held-out test input where you must produce the test output. Grids are 1×1 to 30×30 with up to 10 colors.

**Why it matters:** unlike MMLU/HumanEval/etc., ARC-AGI tasks are *novel* — by design, no two tasks share the same rule. Pure pattern memorization fails. It's one of the few benchmarks where humans (~85%) still trounce frontier LLMs (~80% with extreme reasoning compute, much less for normal use).

**Two versions:**
- **ARC-AGI-1** (2019): 400 train + 400 public eval + 100 semi-private + 100 hidden test. Public eval is what most papers report. Saturating.
- **ARC-AGI-2** (2024): newer, harder, deliberately built to stay hard for LLMs. ~120 public eval at first, growing. This is the *current* leaderboard target.

**Where the official leaderboard lives:** [arcprize.org](https://arcprize.org). Real submissions are scored on the *semi-private* set via a Kaggle competition kernel — no internet, time-limited. We'll cover that flow at the end.

For now we'll measure ourselves on the **public eval** (~400 tasks for v1, ~120 for v2). That's the standard self-reported number, and it's what you'll cite in papers/posts.


## What "running on the benchmark" actually means

Three levels of rigor. Pick whichever you can.

| Level | Set | Internet | Tooling | Score is "official"? |
|---|---|---|---|---|
| **A. Public eval (this notebook)** | 400 tasks ARC1 / ~120 ARC2 | yes | any GPU | self-reported, comparable to papers |
| **B. ARC Prize Kaggle leaderboard** | semi-private 100 | **no internet, 12h time limit** | Kaggle kernel only | yes, public leaderboard rank |
| **C. Hidden test** | 100, never released | no | only via Kaggle staff | only the actual prize winners get this |

For "top 100" you need **B** (Kaggle submission). But the score you get on **A** is a very strong predictor of where you'll land on **B**, so we do **A** first and submit to **B** when it's looking good.


In [ ]:
# 0.1 — Hardware check
import torch, sys, time, os, json
from pathlib import Path
assert torch.cuda.is_available(), "GPU required. Switch Colab runtime to T4/L4/H100."
GPU = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU} ({VRAM_GB:.0f} GB) | torch {torch.__version__}")


In [ ]:
# 0.2 — Install dependencies
!pip -q install -U \
    "transformers>=4.51,<4.55" "peft>=0.14" "accelerate>=1.4" \
    "bitsandbytes>=0.45" "datasets>=3.2" "plotly>=5.24" \
    sentencepiece huggingface_hub


## Step 1 — Tell the notebook where your adapter lives

Three options, in order of how easy they are. Edit `ADAPTER_SOURCE` below.

**Option A — local path** (default; works if you ran notebook 02 or 05 in the same Colab session):
```python
ADAPTER_SOURCE = ("local", "/content/runs/<RUN_ID>/adapters/grpo")
```

**Option B — HuggingFace Hub** (you pushed it; works from anywhere):
```python
ADAPTER_SOURCE = ("hub", "your-handle/qwen3-8b-arc-grpo")
```

**Option C — uploaded zip bundle** (you saved `<RUN_ID>.zip` from notebook 05 and uploaded it to Colab):
```python
ADAPTER_SOURCE = ("zip", "/content/run-1234567890.zip", "adapters/grpo")
```

The base model must match what you trained on. If you used notebook 05's defaults that's `Qwen/Qwen3-8B`.


In [ ]:
# 1.1 — Configure adapter source + base model
BASE_MODEL_ID = "Qwen/Qwen3-8B"

# === EDIT THIS ===
ADAPTER_SOURCE = ("local", "./runs/latest/adapters/grpo")
# Examples:
# ADAPTER_SOURCE = ("hub",   "your-handle/qwen3-8b-arc-grpo")
# ADAPTER_SOURCE = ("zip",   "/content/run-1234567890.zip", "adapters/grpo")
# ADAPTER_SOURCE = ("none",) # eval the base model with no adapter (sanity check)

# Auto-discover most recent run if you used notebook 05 in the same env
if ADAPTER_SOURCE[0] == "local" and not Path(ADAPTER_SOURCE[1]).exists():
    runs_dir = Path("/content/runs") if Path("/content/runs").exists() else Path("./runs")
    if runs_dir.exists():
        latest = sorted(runs_dir.glob("run-*"), key=lambda p: p.stat().st_mtime)
        for cand in reversed(latest):
            for stage in ["grpo", "dpo", "sft"]:
                p = cand / "adapters" / stage
                if p.exists():
                    ADAPTER_SOURCE = ("local", str(p))
                    print(f"Auto-found: {p}")
                    break
            if ADAPTER_SOURCE[1] != "./runs/latest/adapters/grpo": break

print("Adapter source:", ADAPTER_SOURCE)
print("Base model    :", BASE_MODEL_ID)


In [ ]:
# 1.2 — Resolve the adapter into a usable local directory
from huggingface_hub import snapshot_download
import zipfile, shutil

ADAPTER_DIR = None
kind = ADAPTER_SOURCE[0]
if kind == "local":
    ADAPTER_DIR = Path(ADAPTER_SOURCE[1])
    assert ADAPTER_DIR.exists(), f"Local adapter not found: {ADAPTER_DIR}"
elif kind == "hub":
    ADAPTER_DIR = Path(snapshot_download(repo_id=ADAPTER_SOURCE[1]))
elif kind == "zip":
    z, sub = ADAPTER_SOURCE[1], ADAPTER_SOURCE[2]
    extract_to = Path("/tmp/arc_adapter_unzip"); extract_to.mkdir(exist_ok=True)
    with zipfile.ZipFile(z) as zf: zf.extractall(extract_to)
    ADAPTER_DIR = extract_to / sub
    assert ADAPTER_DIR.exists(), f"{sub} not found inside {z}"
elif kind == "none":
    ADAPTER_DIR = None
else:
    raise ValueError(f"Unknown adapter source kind: {kind}")

print("Adapter dir:", ADAPTER_DIR)


## Step 2 — Get the official ARC-AGI eval data

We pull both ARC-AGI-1 and (optionally) ARC-AGI-2. The repos are public on GitHub; we just zip-download them.


In [ ]:
# 2.1 — Download ARC-AGI-1 + (try) ARC-AGI-2
import urllib.request, zipfile, io
DATA_DIR = Path("./arc_data"); DATA_DIR.mkdir(exist_ok=True)

def download_repo(url, target_subdir):
    target = DATA_DIR / target_subdir
    if target.exists() and any(target.glob("**/*.json")):
        return True
    try:
        with urllib.request.urlopen(url) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)
        return True
    except Exception as e:
        print(f"  [skip] {url}: {e}")
        return False

# ARC-AGI-1
download_repo("https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip", "ARC-AGI-master")
v1_eval_dir = DATA_DIR / "ARC-AGI-master" / "data" / "evaluation"

# ARC-AGI-2 (try a few candidate URLs)
v2_eval_dir = None
for url, subdir in [
    ("https://github.com/arcprize/ARC-AGI-2/archive/refs/heads/main.zip", "ARC-AGI-2-main"),
    ("https://github.com/arcprize/ARC-AGI-2/archive/refs/heads/master.zip", "ARC-AGI-2-master"),
]:
    if download_repo(url, subdir):
        cand = DATA_DIR / subdir / "data" / "evaluation"
        if cand.exists() and any(cand.glob("*.json")):
            v2_eval_dir = cand; break

print("ARC-AGI-1 eval dir:", v1_eval_dir, "exists:", v1_eval_dir.exists())
print("ARC-AGI-2 eval dir:", v2_eval_dir, "exists:", v2_eval_dir is not None and v2_eval_dir.exists())


In [ ]:
# 2.2 — Choose which benchmark you want to score on
# v1 is the classic 400-task eval; v2 is the newer harder one. You can run both.
EVAL_VERSION = "v1"   # change to "v2" if v2 downloaded successfully

eval_dir = v1_eval_dir if EVAL_VERSION == "v1" else v2_eval_dir
assert eval_dir is not None and eval_dir.exists(), f"{EVAL_VERSION} eval dir missing"
EVAL_TASKS = [(p.stem, json.loads(p.read_text())) for p in sorted(eval_dir.glob("*.json"))]
print(f"Loaded {len(EVAL_TASKS)} eval tasks from ARC-AGI-{EVAL_VERSION[1:]}")


## Step 3 — Load base model + your adapter

4-bit base + LoRA on top. Same recipe as notebook 02. On a T4 this loads in ~90 seconds.


In [ ]:
# 3.1 — Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.bfloat16,
                        bnb_4bit_use_double_quant=True)

attn_impl = "flash_attention_2" if ("H100" in GPU or "A100" in GPU) else "sdpa"

tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
QWEN3_KW = {"enable_thinking": False} if "Qwen3" in BASE_MODEL_ID else {}

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb, device_map={"": 0},
    torch_dtype=torch.bfloat16, attn_implementation=attn_impl, trust_remote_code=True)

if ADAPTER_DIR is not None:
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False)
    print(f"Loaded base + adapter from {ADAPTER_DIR}")
else:
    model = base_model
    print("Loaded base model only (no adapter — sanity-check mode)")
model.eval()


In [ ]:
# 3.2 — Format helpers (must mirror training notebooks exactly)
SYSTEM = ("You are an expert ARC puzzle solver. Output only the test output grid as digits "
          "separated by spaces, one row per line. No explanations.")

def grid_to_str(g): return "\n".join(" ".join(str(c) for c in row) for row in g)

def parse_grid(s):
    rows = []
    for line in s.strip().splitlines():
        line = line.strip()
        if not line: continue
        try: rows.append([int(c) for c in line.split()])
        except: return None
    if not rows or any(len(r) != len(rows[0]) for r in rows): return None
    return rows

def format_user(task):
    parts = ["You are solving an ARC puzzle. Each grid is a 2D array of digits 0-9 (colors).",
             "Find the transformation rule from the input/output examples, then apply it to the test input.", ""]
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"Example {i} Input:");  parts.append(grid_to_str(ex["input"]))
        parts.append(f"Example {i} Output:"); parts.append(grid_to_str(ex["output"]))
        parts.append("")
    parts.append("Test Input:"); parts.append(grid_to_str(task["test"][0]["input"]))
    parts.append(""); parts.append("Test Output:")
    return "\n".join(parts)

def to_chat_prompt(task):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":format_user(task)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **QWEN3_KW)

@torch.inference_mode()
def predict_greedy(task, max_new=2048):
    text = to_chat_prompt(task)
    inp = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return parse_grid(tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True))


## Step 4 — Run the benchmark, fastest → strongest

We run three modes. Each gets its own results file. You decide how much compute to spend.

| Mode | Speed (T4) | Speed (H100) | Expected lift |
|---|---|---|---|
| Greedy | ~30s/task | ~5s/task | baseline |
| D4 vote | ~4 min/task | ~40s/task | +5–10 pts |
| TTT     | ~10 min/task | ~90s/task | +15–25 pts on top |

Default is **greedy + D4 vote**. TTT is opt-in (Section 6). For a quick smoke test, set `LIMIT = 20` to run 20 tasks instead of all 400.


In [ ]:
# 4.1 — Greedy pass on the full eval (or a subset for sanity)
LIMIT = len(EVAL_TASKS)   # set to 20 for quick smoke test, len(EVAL_TASKS) for full run

from tqdm.auto import tqdm
RESULTS = {"greedy": [], "d4vote": [], "ttt": []}

t0 = time.time()
correct = 0
for name, task in tqdm(EVAL_TASKS[:LIMIT], desc="greedy"):
    pred = predict_greedy(task)
    truth = task["test"][0]["output"]
    ok = (pred == truth)
    correct += int(ok)
    RESULTS["greedy"].append({"task": name, "pred": pred, "correct": ok})

elapsed = time.time() - t0
print(f"\nGREEDY: {correct}/{LIMIT} = {correct/LIMIT:.1%}  ({elapsed:.0f}s, {elapsed/LIMIT:.1f}s/task)")


## Step 5 — D4 self-consistency vote

For each test input, apply each of the 8 dihedral symmetries (rotations + reflections), predict on the transformed task, inverse-transform the output, and majority-vote across the 8 predictions. This is the single most useful inference-time trick on ARC.


In [ ]:
# 5.1 — D4 transforms + their inverses
def rot90(g):  return [list(r) for r in zip(*g[::-1])]
def fliplr(g): return [r[::-1] for r in g]
D4 = [
    lambda g: g, rot90,
    lambda g: rot90(rot90(g)),
    lambda g: rot90(rot90(rot90(g))),
    fliplr,
    lambda g: rot90(fliplr(g)),
    lambda g: rot90(rot90(fliplr(g))),
    lambda g: rot90(rot90(rot90(fliplr(g)))),
]
# Inverse mapping (D4[k] composed with D4[INV[k]] = identity).
# Reflections are involutions (their own inverse). Rotations: rot90 ↔ rot270.
INV = [0, 3, 2, 1, 4, 5, 6, 7]

def apply_d4(task, k):
    f = D4[k]
    return {"train":[{"input":f(e["input"]),"output":f(e["output"])} for e in task["train"]],
            "test": [{"input":f(e["input"]),"output":f(e["output"])} for e in task["test"]]}

def grid_to_tuple(g):
    return tuple(tuple(r) for r in g) if g else None

def predict_d4_vote(task):
    counts = {}
    for k in range(8):
        try:
            t_aug = apply_d4(task, k)
            pred = predict_greedy(t_aug)
            if pred is None: continue
            # inverse-transform back
            pred_inv = D4[INV[k]](pred)
            key = grid_to_tuple(pred_inv)
            counts[key] = counts.get(key, 0) + 1
        except Exception:
            continue
    if not counts: return None
    best = max(counts, key=counts.get)
    return [list(r) for r in best]


In [ ]:
# 5.2 — Run D4 voting on the full eval
t0 = time.time()
correct = 0
for name, task in tqdm(EVAL_TASKS[:LIMIT], desc="d4 vote"):
    pred = predict_d4_vote(task)
    truth = task["test"][0]["output"]
    ok = (pred == truth)
    correct += int(ok)
    RESULTS["d4vote"].append({"task": name, "pred": pred, "correct": ok})

elapsed = time.time() - t0
print(f"\nD4 VOTE: {correct}/{LIMIT} = {correct/LIMIT:.1%}  ({elapsed:.0f}s, {elapsed/LIMIT:.1f}s/task)")


## Step 6 (optional) — Test-time training (TTT)

For each test task, fine-tune a small LoRA on its own train examples (with augmentations) for ~20 steps, then predict, then discard. This is the trick everyone in the top of the leaderboard uses. Costs ~2–10 min/task — set `RUN_TTT = True` only if you have the time budget.


In [ ]:
# 6.1 — TTT predictor (opt-in)
RUN_TTT = False   # flip to True if you want the +15–25pt boost

if RUN_TTT:
    from torch.optim import AdamW
    import random, copy

    def aug_task(task, k, color_perm):
        f = D4[k]
        def apply(ex):
            return {"input":  [[color_perm[c] for c in row] for row in f(ex["input"])],
                    "output": [[color_perm[c] for c in row] for row in f(ex["output"])]}
        return {"train":[apply(e) for e in task["train"]],
                "test": [apply(e) for e in task["test"]]}

    def build_ttt_examples(task, n_aug=20, rng=None):
        rng = rng or random.Random(0)
        out = []
        if len(task["train"]) < 2: return out
        for _ in range(n_aug):
            k = rng.randrange(8)
            perm = list(range(10)); rng.shuffle(perm)
            a = aug_task(task, k, perm)
            for held in range(len(a["train"])):
                sub = {"train":[e for j,e in enumerate(a["train"]) if j != held],
                       "test":[a["train"][held]]}
                msgs = [{"role":"system","content":SYSTEM},
                        {"role":"user","content":format_user(sub)},
                        {"role":"assistant","content":grid_to_str(sub["test"][0]["output"])}]
                out.append(tok.apply_chat_template(msgs, tokenize=False, **QWEN3_KW))
        return out

    def ttt_and_predict(task, steps=20, lr=5e-5, n_aug=20):
        # Snapshot trainable LoRA params
        snap = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters(): p.requires_grad = (n in snap)
        model.train()
        texts = build_ttt_examples(task, n_aug=n_aug)
        if not texts:
            model.eval(); return predict_greedy(task)
        opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
        rng = random.Random(0)
        for _ in range(steps):
            t = rng.choice(texts)
            enc = tok(t, return_tensors="pt", truncation=True, max_length=3500).to(model.device)
            out = model(**enc, labels=enc.input_ids.clone())
            out.loss.backward(); opt.step(); opt.zero_grad()
        model.eval()
        pred = predict_greedy(task)
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in snap: p.copy_(snap[n])
        return pred

    t0 = time.time(); correct = 0
    for name, task in tqdm(EVAL_TASKS[:LIMIT], desc="ttt"):
        try: pred = ttt_and_predict(task)
        except Exception: pred = None
        ok = (pred == task["test"][0]["output"])
        correct += int(ok)
        RESULTS["ttt"].append({"task": name, "pred": pred, "correct": ok})
    elapsed = time.time() - t0
    print(f"\nTTT: {correct}/{LIMIT} = {correct/LIMIT:.1%}  ({elapsed:.0f}s, {elapsed/LIMIT:.1f}s/task)")
else:
    print("Skipping TTT (set RUN_TTT=True to enable).")


## Step 7 — Generate the official `submission.json`

The ARC-AGI submission format gives you **2 attempts** per task. Either is allowed to be correct (pass@2). We use:
- `attempt_1` = best mode you ran (TTT > D4 vote > greedy)
- `attempt_2` = second-best mode

This matches the official Kaggle competition format, so you can use this file directly.


In [ ]:
# 7.1 — Build submission.json with the 2 best attempts you have
def best_two_modes():
    if RESULTS["ttt"]:    return ("ttt", "d4vote")
    if RESULTS["d4vote"]: return ("d4vote", "greedy")
    return ("greedy", "greedy")

a1_mode, a2_mode = best_two_modes()
def by_task(mode):
    return {r["task"]: r for r in RESULTS[mode]}
A1 = by_task(a1_mode); A2 = by_task(a2_mode)

submission = {}
for name, task in EVAL_TASKS[:LIMIT]:
    p1 = (A1.get(name) or {}).get("pred") or [[0]]
    p2 = (A2.get(name) or {}).get("pred") or [[0]]
    submission[name] = [{"attempt_1": p1, "attempt_2": p2}]

out_path = Path(f"submission_{EVAL_VERSION}.json")
out_path.write_text(json.dumps(submission))
print(f"Wrote {out_path} | attempt_1={a1_mode}, attempt_2={a2_mode}, tasks={len(submission)}")

# pass@2 score
def pass_at_2():
    correct = 0
    for name, task in EVAL_TASKS[:LIMIT]:
        truth = task["test"][0]["output"]
        if A1.get(name, {}).get("pred") == truth or A2.get(name, {}).get("pred") == truth:
            correct += 1
    return correct, LIMIT

p2c, p2n = pass_at_2()
print(f"pass@2 score: {p2c}/{p2n} = {p2c/p2n:.1%}")


## Step 8 — Inspect what worked and what didn't

A small interactive plotly dashboard. Helps you see whether your model fails on big grids, certain task patterns, etc.


In [ ]:
# 8.1 — Per-task results dashboard
import plotly.graph_objects as go
from plotly.subplots import make_subplots

modes_present = [m for m in ["greedy","d4vote","ttt"] if RESULTS[m]]

# accuracy bars
acc_per_mode = {m: sum(r["correct"] for r in RESULTS[m]) / max(1, len(RESULTS[m])) for m in modes_present}
fig1 = go.Figure(data=[go.Bar(x=list(acc_per_mode.keys()),
                              y=[v*100 for v in acc_per_mode.values()],
                              text=[f"{v:.1%}" for v in acc_per_mode.values()],
                              textposition="outside")])
fig1.update_layout(title=f"Accuracy by mode (ARC-AGI-{EVAL_VERSION[1:]}, n={LIMIT})",
                   yaxis_title="accuracy (%)", template="plotly_white")
fig1.write_html("results_accuracy.html"); fig1.show()


In [ ]:
# 8.2 — Correctness by test-input grid size
def grid_area(g): return len(g) * len(g[0]) if g else 0
def task_size(task): return grid_area(task["test"][0]["input"])

mode = "d4vote" if RESULTS["d4vote"] else "greedy"
sizes_correct = []
sizes_wrong = []
for name, task in EVAL_TASKS[:LIMIT]:
    rec = next((r for r in RESULTS[mode] if r["task"] == name), None)
    if rec is None: continue
    (sizes_correct if rec["correct"] else sizes_wrong).append(task_size(task))

fig2 = go.Figure()
fig2.add_trace(go.Histogram(x=sizes_correct, name="correct", opacity=0.7, nbinsx=30))
fig2.add_trace(go.Histogram(x=sizes_wrong,   name="wrong",   opacity=0.7, nbinsx=30))
fig2.update_layout(barmode="overlay", title=f"Correctness vs test grid area ({mode})",
                   xaxis_title="test input area (cells)", yaxis_title="task count",
                   template="plotly_white")
fig2.write_html("results_by_size.html"); fig2.show()


In [ ]:
# 8.3 — Print 3 successes + 3 failures so you can eyeball them
mode = "d4vote" if RESULTS["d4vote"] else "greedy"
recs = RESULTS[mode]
wins = [r for r in recs if r["correct"]][:3]
fails = [r for r in recs if not r["correct"]][:3]

def show(title, r):
    task = next(t for n,t in EVAL_TASKS if n == r["task"])
    truth = task["test"][0]["output"]
    print(f"\n=== {title}: {r['task']} ===")
    print("test input:");  print(grid_to_str(task["test"][0]["input"]))
    print("truth:");       print(grid_to_str(truth))
    print("model output:"); print(grid_to_str(r["pred"]) if r["pred"] else "<unparseable>")

for r in wins:  show("WIN",  r)
for r in fails: show("FAIL", r)


## Step 9 — (Optional) Host the adapter on HuggingFace Hub

Pushing the LoRA adapter to HF Hub gives you:
- A permanent URL anyone can pull from (`from_pretrained("your-handle/qwen3-8b-arc-grpo")`)
- Free hosting (LoRA adapters are tiny, ~200 MB)
- Versioning + a README

You'll need a HuggingFace account + a write token from https://huggingface.co/settings/tokens


In [ ]:
# 9.1 — Push adapter to HF Hub (one-time)
PUSH_TO_HUB = False           # flip to True after editing the repo id below
HUB_REPO_ID = "your-handle/qwen3-8b-arc-grpo"

if PUSH_TO_HUB and ADAPTER_DIR is not None:
    from huggingface_hub import login, HfApi
    login()                    # paste your write token when prompted
    HfApi().create_repo(HUB_REPO_ID, repo_type="model", exist_ok=True, private=False)
    HfApi().upload_folder(folder_path=str(ADAPTER_DIR),
                          repo_id=HUB_REPO_ID, repo_type="model")
    print(f"Pushed: https://huggingface.co/{HUB_REPO_ID}")
else:
    print("Skipped. Set PUSH_TO_HUB=True and edit HUB_REPO_ID to push.")


## Step 10 — (Optional) Serve the model locally

For a production-style local server, the simplest path is **vLLM**. One command after merging the adapter into the base:

```bash
# 1. Merge adapter into base (one-time, produces a standalone model dir)
python -c "
from peft import PeftModel; from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-8B', torch_dtype='bfloat16')
m = PeftModel.from_pretrained(base, 'PATH/TO/ADAPTER')
m.merge_and_unload().save_pretrained('./qwen3-8b-arc-merged')
"

# 2. Start the server
pip install vllm
vllm serve ./qwen3-8b-arc-merged --port 8000

# 3. Hit it like an OpenAI-compatible endpoint:
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen3-8b-arc-merged","messages":[{"role":"user","content":"..."}]}'
```

Skip this if you only care about getting a benchmark score — the eval above already runs the model directly.


## Step 11 — Submit to the official ARC Prize leaderboard

The arcprize.org leaderboard is hosted as a **Kaggle competition** with these constraints:

1. Submission is a Kaggle **kernel** (notebook), not a JSON upload.
2. Kernel has **no internet** at submission time.
3. ~12 hours of runtime budget on the Kaggle hardware (T4 x 2).

**The flow:**

1. Go to the active ARC Prize competition on Kaggle (search "ARC Prize" on kaggle.com/competitions).
2. Upload your **adapter** as a Kaggle Dataset (private is fine). Get the dataset slug.
3. Upload your **base model** as a Kaggle Dataset too (or use a Kaggle-hosted one if available). Internet-off kernels can't `from_pretrained` from HF Hub.
4. Fork the competition's starter kernel. Replace its inference cell with the equivalent of Steps 4–7 of this notebook (greedy + D4 vote, TTT if time permits).
5. **Set internet OFF, GPU ON, accelerator T4 x 2** in kernel settings.
6. "Save & Run All" to commit a version. Submit that version.
7. Wait for the score to appear under your team on the leaderboard.

**Common gotchas:**
- The kernel must finish in budget — TTT on all 100 tasks at 10 min each is 16+ hours, won't fit. Tune `steps` and `n_aug` down.
- The kernel must produce `submission.json` at the working directory root.
- Models loaded via `from_pretrained` need the dataset path (e.g. `/kaggle/input/<your-dataset>/`).
- The grid format must match exactly (digits 0–9 separated by spaces, one row per line; or the JSON list-of-lists in the submission file).


## FAQ / Common gotchas

**Q: I got 0% accuracy.**
A: Almost certainly a prompt-format mismatch between training and eval. Confirm `SYSTEM`, `format_user`, and `enable_thinking=False` (for Qwen3) match exactly between this notebook and the training notebook.

**Q: My T4 OOMs loading 8B in 4-bit.**
A: Lower `MAX_LEN` (cell 3.2), or drop to `Qwen/Qwen3-4B`. T4 has 16 GB which is tight for 8B with KV cache.

**Q: D4 vote made my score *worse*.**
A: Means your model is overfit to D4-augmented training and produces inconsistent outputs across rotations. Mostly happens with very short SFT runs. Use greedy as `attempt_1` instead.

**Q: TTT is too slow on my GPU.**
A: Drop `steps=20` to `steps=10` and `n_aug=20` to `n_aug=10`. Or skip TTT entirely — strong SFT + D4 vote alone is enough for a respectable public-eval score.

**Q: Where do I see my official rank?**
A: Only after a Kaggle submission (Step 11). Public eval is for self-reporting and dev iteration.

**Q: Why don't I match the top of the leaderboard?**
A: The very top entries use 70B+ models, custom solver scaffolds, very long TTT, and ensembles of multiple checkpoints. A single H100 + single 7B/8B + LoRA + TTT typically lands in the **15–35% range on ARC-AGI-1 public eval**, which is solid and comparable to most published baselines.
